In [1]:
import pandas as pd
import matplotlib.pyplot as plt
from scipy.stats import linregress, t
from scipy import stats
import numpy as np
from pathlib import Path
from scipy.optimize import curve_fit
import re
import os

# Statements' classification

In [3]:
scenario_0 = "0-random both repeated and non repeated can be true or false"


In [3]:
statements = pd.read_excel("data/Griffin_et_al/statements.xls").rename(
    columns={"id": "statement_id", "text": "statement"})
for i in range(1, 4):
    with open("data/Griffin_et_al/statement_classifications_trial_" + str(i) + ".txt", "r") as f:
        classification = f.readlines()
    classification = [True if line.split('":')[1].split("-")[0].strip() == "True" else False for line in
                      classification]
    statements["type_trial_" + str(i)] = classification

statements["agreement"] = statements.apply(lambda x: x.type_trial_1 == x.type_trial_2 == x.type_trial_3, axis=1)

In [127]:
# dropping some opinion-like statements
ids_to_drop = [73, 34, 63, 76, 57, 70, 96, 36, 41, 92]
df_clean_up = statements[~statements["statement_id"].isin(ids_to_drop)]
df = df_clean_up[(df_clean_up["agreement"] == False)][["statement_id", "statement"]]
df

,statement_id,statement
12,13,Aluminium is soluble in salt water
38,39,The cucumovirus has dodecahedral symmetry
39,40,Bulging eyes are a symptom of high blood pressure
47,48,Caledonian Road junctions with Holloway and Pe...
60,61,Nematode worms knot themselves around prey
67,68,Millions of children die annually through hous...
68,69,Piet van der Schans was an olympican equestrian
98,99,Same sex marriage is legal for 1/3 of the worl...


In [128]:
# Manually classifying statements on which Claude didn't agree over three trials.
df["type"] = [False, False, True, False, False, False, True, False]
statements_class = pd.concat([df, df_clean_up[df_clean_up["agreement"] == True][
    ["statement_id", "statement", "type_trial_1"]].rename(columns={"type_trial_1": "type"})])
data = statements[["statement_id", "statement"]].merge(statements_class[["statement_id", "type"]],how="left", on="statement_id")
#data.to_csv("data/statements.csv", index = False)

In [129]:
data = pd.read_csv("data/statements.csv")

# Helper functions

In [38]:
# Define the model from Equation 1: r' = r + offset + tilt × (r - 4)
def model_eq1(r, offset, tilt):
    return r + offset + tilt * (r - 4)

def get_stats(x, y, confidence_level=0.95):
    # Fit the model y = ax + b
    slope, intercept, r_value_reg, p_value_zero, std_err_reg = linregress(x, y)
    df = len(x) - 2
    t_stat_one = (slope - 1) / std_err_reg
    p_value_one = 2 * t.sf(np.abs(t_stat_one), df)

    # Fit the model: r' = r + offset + tilt × (r - 4)
    params, covariance = curve_fit(model_eq1, x, y)
    offset, tilt = params

    # Get standard errors
    perr = np.sqrt(np.diag(covariance))
    offset_se, tilt_se = perr

    # Calculate t-statistics and p-values
    t_offset = offset / offset_se
    t_tilt = tilt / tilt_se
    p_offset = 2 * (1 - stats.t.cdf(abs(t_offset), len(x) - 2))
    p_tilt = 2 * (1 - stats.t.cdf(abs(t_tilt), len(x) - 2))

    # Calculate confidence intervals using t-distribution
    alpha = 1 - confidence_level
    t_critical = stats.t.ppf(1 - alpha/2, df=len(x) - 2)
    
    # Confidence interval half-width (margin of error)
    offset_ci_margin = t_critical * offset_se
    tilt_ci_margin = t_critical * tilt_se
    
    # Full confidence intervals
    offset_ci = (offset - offset_ci_margin, offset + offset_ci_margin)
    tilt_ci = (tilt - tilt_ci_margin, tilt + tilt_ci_margin)

    # Print results
    print(f"offset = {offset:.3f} (SE = {offset_se:.3f}, t = {t_offset:.3f}, p = {p_offset:.4f})")
    print(f"  95% CI: [{offset_ci[0]:.3f}, {offset_ci[1]:.3f}]")
    print(f"tilt = {tilt:.3f} (SE = {tilt_se:.3f}, t = {t_tilt:.3f}, p = {p_tilt:.4f})")
    print(f"  95% CI: [{tilt_ci[0]:.3f}, {tilt_ci[1]:.3f}]")

    # Predicted values
    y_pred = model_eq1(x, offset, tilt)

    # Calculate R²
    ss_res = np.sum((y - y_pred)**2)
    ss_tot = np.sum((y - np.mean(y))**2)
    r_squared_eq_1 = 1 - (ss_res / ss_tot)
    print(f"R² = {r_squared_eq_1:.3f}")
    
    return offset, tilt, p_offset, p_tilt, r_squared_eq_1, slope, intercept, r_value_reg, p_value_one, std_err_reg, offset_ci, tilt_ci

In [53]:
def get_rating_results(x, y, err_x, err_y, model="GPT-5-nano", attribute="truth", plot=False, eq="eq_1", temperature=1):
    # Calculate and plot regression line
    offset, tilt, p_offset, p_tilt, r_squared_eq_1, slope, intercept, r_value_reg, p_value_one, std_err_reg, offset_ci, tilt_ci = get_stats(
        x, y)

    FONT_SIZE = 26

    # Create figure and gridspec
    fig = plt.figure(figsize=(10, 10))
    gs = fig.add_gridspec(2, 2, width_ratios=(1, 4), height_ratios=(4, 1),
                          wspace=0.05, hspace=0.05)

    ax_scatter = fig.add_subplot(gs[0, 1])

    # Create scatter plot
    ax_scatter.errorbar(x, y, fmt='o', alpha=0.5, xerr=err_x, yerr=err_y, ecolor='gray', capsize=4,
                        markersize=10, elinewidth=2, capthick=2)

    if x.max() > 4:
        ax_scatter.set_xticks(range(8))
        ax_scatter.set_yticks(range(8))
        ax_scatter.set_xlim(0.5, 7.5)
        ax_scatter.set_ylim(0.5, 7.5)
    else:
        ax_scatter.set_xticks(range(5))
        ax_scatter.set_yticks(range(5))
        ax_scatter.set_xlim(-0.5, 4.5)
        ax_scatter.set_ylim(-0.5, 4.5)

    ax_scatter.tick_params(axis='both', labelsize=FONT_SIZE)

    line_x = np.linspace(x.min(), x.max(), 100)
    if eq == "eq_reg":
        line_y = slope * line_x + intercept
        ax_scatter.plot(line_x, line_y, 'r-', label=f'Regression line (R² = {r_value_reg ** 2:.3f})')
    else:
        line_y = model_eq1(line_x, offset, tilt)
        ax_scatter.plot(line_x, line_y, 'r-', label=f'Regression line Eq 1 (R² = {r_squared_eq_1:.3f})')

    # Add dashed x = y line
    ax_scatter.plot([0, 8], [0, 8], 'k--', alpha=0.7, label='x = y')

    # Add labels and title
    ax_scatter.set_xlabel('Mean ' + attribute + ' rating unseen', fontsize=FONT_SIZE)
    ax_scatter.set_ylabel('Mean ' + attribute + ' rating repeated', fontsize=FONT_SIZE)
    # ax_scatter.legend(fontsize=FONT_SIZE)
    ax_scatter.grid(True, alpha=0.3)

    # Save the figure BEFORE showing it
    plot_folder = f"experiments/plots/"
    os.makedirs(plot_folder, exist_ok=True)
    filename = f"{model}_temp_{temperature}_{attribute}.pdf"
    plt.savefig(plot_folder + filename, dpi=300, bbox_inches='tight')
    if not plot:
        plt.close(fig)

    # Format confidence intervals as strings with 3 decimal places
    offset_ci_str = f"[{offset_ci[0]:.3f}, {offset_ci[1]:.3f}]"
    tilt_ci_str = f"[{tilt_ci[0]:.3f}, {tilt_ci[1]:.3f}]"

    # Save the regression statistics
    return pd.DataFrame({"offset": [f"{offset:.3f}"],
                         "offset_CI": offset_ci_str,
                         "tilt": f"{tilt:.3f}",
                         "tilt_CI": tilt_ci_str,
                         "p_offset": f"{p_offset:.3f}",
                         "p_tilt": f"{p_tilt:.3f}",
                         "r_squared_eq_1": f"{r_squared_eq_1:.3f}",
                         "slope": f"{slope: .4f}",
                         "intercept": f"{intercept: .4f}",
                         "R-squared": f"{r_value_reg ** 2:.4f}",
                         "P-value (vs 1)": f"{p_value_one:.4f}"})


In [40]:
def get_ci(data):
    # Drop NaNs to ensure we calculate stats on valid ratings only
    data = data.dropna()
    # We need at least 2 samples to calculate variance (n-1 degrees of freedom)
    if len(data) < 2:
        return np.nan

    # Compute 95% confidence interval
    confidence_level = 0.95
    mean = np.mean(data)
    sem = stats.sem(data)
    if sem == 0:
        # No variability: CI collapses to the mean
        ci = (mean, mean)
        ci_half_width = 0
    else:
        ci = stats.t.interval(confidence_level,
                              df=len(data)-1,  # degrees of freedom
                              loc=mean,         # mean
                              scale=sem)  # standard error

        # ci is a tuple: (lower_bound, upper_bound)
        ci_lower, ci_upper = ci
        ci_half_width = ci_upper - mean  # for error bars
    return ci_half_width

In [43]:
def get_rating_statement_open_weights(claims_ratings, attribute = "truth"):
    claims_ratings = claims_ratings[claims_ratings["iteration_num"].isin(["claim", "rating"])]
    ratings = []
    statement = ""
    statement_id = ""
    target = ""
    type = ""
    for line in claims_ratings.itertuples():
        if line.role == "user":
            statement = line.content
            statement_id = line.statement_id
            target = True if (line.target == True) else False
            type = line.type
        else:
            ratings.append(pd.DataFrame({"simulation_num": line.simulation_num,
                                               "trial_num": line.trial_num,
                                               "iteration_num": line.iteration_num,
                                               "statement_id": statement_id,
                                               "statement": [statement],
                                               "target": target,
                                               "type": type,
                                               attribute + "_rating": line.content
                                               }))
    ratings = pd.concat(ratings)
    ratings[attribute + "_rating"] = ratings[attribute + "_rating"].apply(lambda x: get_rating_string(x))
    ratings[attribute + "_rating"] = pd.to_numeric(ratings[attribute + "_rating"], errors='coerce')
    ratings = ratings.reset_index(drop = True)
    mean_ratings = ratings.groupby(["target", "statement_id"])[attribute + "_rating"].mean().reset_index().rename(columns={attribute + "_rating": "mean_" + attribute + "_rating"})
    std_ratings = ratings.groupby(["target", "statement_id"])[attribute + "_rating"].std().reset_index().rename(columns={attribute + "_rating": "std_" + attribute + "_rating"})
    err_ratings = ratings.groupby(["target", "statement_id"])[attribute + "_rating"].apply(lambda x: get_ci(x)).reset_index().rename(columns={attribute + "_rating": "err_" + attribute + "_rating"})
    df = mean_ratings.merge(std_ratings, on = ["target", "statement_id"]).merge(err_ratings, on = ["target", "statement_id"])
    return df

In [44]:
def plot_std(std, model, temperature, attribute, repeated = None):
    data = pd.Series(std).dropna()
    if len(data) == 0:
        return
    plt.figure(figsize=(10, 6))
    plt.hist(data, bins=20, density=True, alpha=0.6, color='skyblue', edgecolor='black', label='Histogram')

    if len(data) > 1 and data.std() > 0:
        try:
            kde = stats.gaussian_kde(data)
            x_vals = np.linspace(data.min(), data.max(), 200)
            plt.plot(x_vals, kde(x_vals), 'r-', linewidth=2, label='KDE')
        except Exception:
            pass
    if repeated:
        rep = "repeated" if repeated else "unseen"
    else:
        rep = "all"
    plt.title(f"Density distribution of std of ratings of {rep} claims\nModel: {model}, Temperature: {temperature}, Attribute: {attribute}")
    plt.xlabel("Standard Deviation")
    plt.ylabel("Density")
    plt.legend()
    plt.grid(True, alpha=0.3)

    plot_folder = "experiments/plots/std_plots/"
    os.makedirs(plot_folder, exist_ok=True)
    filename = f"std_density_{model}_temp_{temperature}_{attribute}_{rep}.pdf"
    plt.savefig(plot_folder + filename, dpi=300, bbox_inches='tight')
    plt.close()

In [76]:
def get_most_recent_results(model = "gpt-5-nano", temperature = 1.0, attribute = "truth"):
    if "gpt" in model:
        path = Path(f"experiments/{model}/{attribute}")
        path = sorted([f for f in path.rglob("*ratings.csv") if f.is_file()])[-1]
        date = re.findall(r"\d{4}_\d{2}_\d{2}_\d{2}_\d{2}", str(path))[0]
        # print(path)
        return {"path": path, "date": date}
    path = Path(f"experiments/{model}/temp_{str(temperature)}/{attribute}/")
    path = sorted([f for f in path.rglob("*simulation_results.csv") if f.is_file()])[-1]
    date = re.findall(r"\d{4}_\d{2}_\d{2}_\d{2}_\d{2}", str(path))[0]
    # print(path)
    return {"path": path, "date": date}
tmp = pd.read_csv(get_most_recent_results(model="Meta-Llama-3.1-8B-Instruct")["path"])
tmp2 = pd.read_csv(get_most_recent_results(model="gpt-5-nano")["path"])

In [46]:
def get_rating_string(rating):
    match = re.search(r"[1-7]", rating)
    if match:
        return match.group()
    return None

In [64]:
def get_results_open_weights(model, attribute, temperature = 1.0):
    df_results = []
    print("attribute ", attribute)
    results = pd.read_csv(get_most_recent_results(model = model, attribute=attribute, temperature=temperature)["path"], dtype={"rating": str, "truth_rating": str}).rename(columns={"truth_rating": "rating"})
    ratings = get_rating_statement_open_weights(results, attribute=attribute)
    unseen = ratings[ratings["target"] == False]
    repeated = ratings[ratings["target"] == True]

    plot_df_open_weight = repeated.merge(unseen, on="statement_id", how = "left", suffixes=("_repeated", "_unseen"))
    plot_df_open_weight = plot_df_open_weight.dropna()
    df_results.append(get_rating_results(plot_df_open_weight["mean_" + attribute + "_rating_unseen"], plot_df_open_weight["mean_" + attribute + "_rating_repeated"],
                      plot_df_open_weight["err_" + attribute + "_rating_unseen"], plot_df_open_weight["err_" + attribute + "_rating_repeated"],
                      model=model, attribute=attribute, temperature=temperature))
    print("len(x)", len(plot_df_open_weight["mean_" + attribute + "_rating_unseen"]))
    print("len(y)", len(plot_df_open_weight["mean_" + attribute + "_rating_repeated"]))
    df_results = pd.concat(df_results)
    df_results["attribute"] = attribute
    return df_results, plot_df_open_weight

In [50]:
def get_baseline_GPT(attribute = "truth"):
    results_GPT = pd.read_csv(get_most_recent_results(model = "gpt-5-nano", attribute=attribute)["path"])
    results_GPT["rating"] = pd.to_numeric(results_GPT["rating"], errors='coerce')
    # by design, every repeated statement is rated at least 30 times (3 trials x 10 simulations per statement)
    rating_unseen = results_GPT[results_GPT["target"] == False].groupby("statement_id")["rating"].mean().reset_index().rename(columns = {"rating": "mean_" + attribute + "_rating"}).merge(results_GPT[results_GPT["target"] == False].groupby("statement_id")["rating"].std().reset_index().rename(columns = {"rating": "std_" + attribute + "_rating"}))
    ids = set(rating_unseen["statement_id"])
    assert len(ids) == 100, "missing unseen ratings for GPT + attribute " + attribute
    return rating_unseen
tmp = get_baseline_GPT()

In [51]:
def prepare_data_gpt_exp(results, attribute = "truth"):
    rating_unseen = results[results["target"] == False].groupby("statement_id")["rating"].mean().reset_index().rename(columns = {"rating": "mean_" + attribute + "_rating"}).merge(results[results["target"] == False].groupby("statement_id")["rating"].std().reset_index().rename(columns = {"rating": "std_" + attribute + "_rating"})).merge(results[results["target"] == False].groupby("statement_id")["rating"].apply(lambda x: get_ci(x)).reset_index().rename(columns = {"rating": "err_" + attribute + "_rating"}))
    rating_repeated = results[results["target"] == True].groupby("statement_id")["rating"].mean().reset_index().rename(columns = {"rating": "mean_" + attribute + "_rating"}).merge(results[results["target"] == True].groupby("statement_id")["rating"].std().reset_index().rename(columns = {"rating": "std_" + attribute + "_rating"})).merge(results[results["target"] == True].groupby("statement_id")["rating"].apply(lambda x: get_ci(x)).reset_index().rename(columns = {"rating": "err_" + attribute + "_rating"}))
    return rating_repeated.merge(rating_unseen, how = "left", on="statement_id", suffixes =("_repeated", "_unseen"))

In [57]:
def get_results_GPT(attribute):
    print(attribute)
    baseline = get_baseline_GPT(attribute)
    df_results = []
    results_GPT = pd.read_csv(get_most_recent_results(model = "gpt-5-nano", attribute=attribute)["path"], dtype={"rating": str, "truth_rating": str}).rename(columns={"truth_rating": "rating"})
    # this leads to errors with GPT answers
    # results_GPT["rating"] = results_GPT["rating"].apply(lambda x: get_rating_string(x))
    results_GPT["rating"] = pd.to_numeric(results_GPT["rating"], errors='coerce')
    results_GPT = results_GPT.dropna(subset=["rating"])
    rating_statements_GPT = prepare_data_gpt_exp(results_GPT, attribute = attribute)
    print("len(data) = ", len(rating_statements_GPT))
    df_results.append(get_rating_results(rating_statements_GPT["mean_" + attribute + "_rating_unseen"], rating_statements_GPT["mean_" + attribute + "_rating_repeated"], rating_statements_GPT["err_" + attribute + "_rating_unseen"], rating_statements_GPT["err_" + attribute + "_rating_repeated"], model = "GPT-5-nano", attribute=attribute))
    # break
    df_results = pd.concat(df_results)
    df_results["attribute"] = attribute
    return df_results, rating_statements_GPT

tmp = get_results_GPT("truth")

truth
len(data) =  100
offset = -0.063 (SE = 0.042, t = -1.484, p = 0.1410)
  95% CI: [-0.147, 0.021]
tilt = 0.001 (SE = 0.019, t = 0.051, p = 0.9595)
  95% CI: [-0.036, 0.038]
R² = 0.967


# GPT-5-nano

In [59]:
nbr_sim = 1000
results_GPT = []
for attribute in ["truth", "importance", "sentiment", "interest"]: #, "importance", "sentiment", "interest"
    results_GPT.append(get_results_GPT(attribute)[0])
results_GPT = pd.concat(results_GPT)
os.makedirs(f"experiments/stats_results/", exist_ok=True)
results_GPT.to_csv(f"experiments/stats_results/gpt-5-nano_temp_1.csv", index = False)

truth
len(data) =  100
offset = -0.063 (SE = 0.042, t = -1.484, p = 0.1410)
  95% CI: [-0.147, 0.021]
tilt = 0.001 (SE = 0.019, t = 0.051, p = 0.9595)
  95% CI: [-0.036, 0.038]
R² = 0.967
importance
len(data) =  100
offset = 0.003 (SE = 0.036, t = 0.095, p = 0.9246)
  95% CI: [-0.069, 0.076]
tilt = -0.163 (SE = 0.027, t = -5.982, p = 0.0000)
  95% CI: [-0.217, -0.109]
R² = 0.906
sentiment
len(data) =  100
offset = -0.063 (SE = 0.018, t = -3.519, p = 0.0007)
  95% CI: [-0.098, -0.027]
tilt = -0.126 (SE = 0.017, t = -7.526, p = 0.0000)
  95% CI: [-0.160, -0.093]
R² = 0.965
interest
len(data) =  100
offset = -0.113 (SE = 0.032, t = -3.527, p = 0.0006)
  95% CI: [-0.176, -0.049]
tilt = -0.253 (SE = 0.031, t = -8.046, p = 0.0000)
  95% CI: [-0.315, -0.191]
R² = 0.852


In [60]:
nbr_sim = 1000
results_GPT = []
for attribute in ["truth", "importance", "sentiment", "interest"]:
    plot_std(get_results_GPT(attribute)[1]["std_" + attribute + "_rating_unseen"], model = "GPT", temperature=1, repeated = True, attribute= attribute)
    plot_std(get_results_GPT(attribute)[1]["std_" + attribute + "_rating_repeated"], model = "GPT", temperature=1, repeated = False, attribute= attribute)


truth
len(data) =  100
offset = -0.063 (SE = 0.042, t = -1.484, p = 0.1410)
  95% CI: [-0.147, 0.021]
tilt = 0.001 (SE = 0.019, t = 0.051, p = 0.9595)
  95% CI: [-0.036, 0.038]
R² = 0.967
truth
len(data) =  100
offset = -0.063 (SE = 0.042, t = -1.484, p = 0.1410)
  95% CI: [-0.147, 0.021]
tilt = 0.001 (SE = 0.019, t = 0.051, p = 0.9595)
  95% CI: [-0.036, 0.038]
R² = 0.967
importance
len(data) =  100
offset = 0.003 (SE = 0.036, t = 0.095, p = 0.9246)
  95% CI: [-0.069, 0.076]
tilt = -0.163 (SE = 0.027, t = -5.982, p = 0.0000)
  95% CI: [-0.217, -0.109]
R² = 0.906
importance
len(data) =  100
offset = 0.003 (SE = 0.036, t = 0.095, p = 0.9246)
  95% CI: [-0.069, 0.076]
tilt = -0.163 (SE = 0.027, t = -5.982, p = 0.0000)
  95% CI: [-0.217, -0.109]
R² = 0.906
sentiment
len(data) =  100
offset = -0.063 (SE = 0.018, t = -3.519, p = 0.0007)
  95% CI: [-0.098, -0.027]
tilt = -0.126 (SE = 0.017, t = -7.526, p = 0.0000)
  95% CI: [-0.160, -0.093]
R² = 0.965
sentiment
len(data) =  100
offset = -0.0

# open weight models

In [65]:
nbr_sim = 1000
for model in ["Meta-Llama-3.1-8B-Instruct", "gemma-3-4b-it", "Qwen2.5-7B-Instruct"]:
    print("\n****************")
    print(model)
    for temp in [1.0, 0.1]:
        print("Temperature", temp)
        print("****************")
        results = []
        for attribute in ["truth", "importance", "sentiment", "interest"]: #, "importance", "sentiment", "interest"
            results.append(get_results_open_weights(model=model, attribute=attribute,  temperature = temp)[0])
        results = pd.concat(results)
        results.to_csv(f"experiments/stats_results/{model}_temp_{temp}.csv", index = False)


****************
Meta-Llama-3.1-8B-Instruct
Temperature 1.0
****************
attribute  truth
offset = -1.436 (SE = 0.264, t = -5.450, p = 0.0000)
  95% CI: [-1.959, -0.913]
tilt = -0.948 (SE = 0.173, t = -5.482, p = 0.0000)
  95% CI: [-1.291, -0.605]
R² = 0.001
len(x) 100
len(y) 100
attribute  importance
offset = -1.910 (SE = 0.254, t = -7.509, p = 0.0000)
  95% CI: [-2.415, -1.405]
tilt = -1.260 (SE = 0.167, t = -7.538, p = 0.0000)
  95% CI: [-1.592, -0.928]
R² = 0.024
len(x) 100
len(y) 100
attribute  sentiment
offset = -1.750 (SE = 0.228, t = -7.673, p = 0.0000)
  95% CI: [-2.202, -1.297]
tilt = -1.152 (SE = 0.150, t = -7.686, p = 0.0000)
  95% CI: [-1.450, -0.855]
R² = 0.010
len(x) 100
len(y) 100
attribute  interest
offset = -1.808 (SE = 0.260, t = -6.948, p = 0.0000)
  95% CI: [-2.324, -1.292]
tilt = -1.192 (SE = 0.171, t = -6.982, p = 0.0000)
  95% CI: [-1.531, -0.854]
R² = 0.013
len(x) 100
len(y) 100
Temperature 0.1
****************
attribute  truth
offset = -1.627 (SE = 0.226,

In [88]:
nbr_sim = 1000
for model in ["Meta-Llama-3.1-8B-Instruct", "gemma-3-4b-it", "Qwen2.5-7B-Instruct"]:
    print("\n****************")
    print(model)
    for temp in [1.0, 0.1]: #1.0
        print("Temperature", temp)
        print("****************")
        results = []
        for attribute in ["truth", "importance", "sentiment", "interest"]:
            print(attribute)
            df = get_results_open_weights(model=model, attribute=attribute, temperature = temp)[1]
            plot_std(df["std_" + attribute + "_rating_unseen"], model = model, temperature=temp, attribute = attribute, repeated=False)
            plot_std(df["std_" + attribute + "_rating_repeated"], model = model, temperature=temp, attribute = attribute, repeated=True)


****************
Meta-Llama-3.1-8B-Instruct
Temperature 1.0
****************
truth
attribute  truth
offset = -1.436 (SE = 0.264, t = -5.450, p = 0.0000)
  95% CI: [-1.959, -0.913]
tilt = -0.948 (SE = 0.173, t = -5.482, p = 0.0000)
  95% CI: [-1.291, -0.605]
R² = 0.001
len(x) 100
len(y) 100
importance
attribute  importance
offset = -1.910 (SE = 0.254, t = -7.509, p = 0.0000)
  95% CI: [-2.415, -1.405]
tilt = -1.260 (SE = 0.167, t = -7.538, p = 0.0000)
  95% CI: [-1.592, -0.928]
R² = 0.024
len(x) 100
len(y) 100
sentiment
attribute  sentiment
offset = -1.750 (SE = 0.228, t = -7.673, p = 0.0000)
  95% CI: [-2.202, -1.297]
tilt = -1.152 (SE = 0.150, t = -7.686, p = 0.0000)
  95% CI: [-1.450, -0.855]
R² = 0.010
len(x) 100
len(y) 100
interest
attribute  interest
offset = -1.808 (SE = 0.260, t = -6.948, p = 0.0000)
  95% CI: [-2.324, -1.292]
tilt = -1.192 (SE = 0.171, t = -6.982, p = 0.0000)
  95% CI: [-1.531, -0.854]
R² = 0.013
len(x) 100
len(y) 100
Temperature 0.1
****************
truth
att

# Output file for linear regression

In [67]:
path = Path(f"experiments/stats_results")
results_table = []
for temp in [0.1, 1.0]:
    temp_results = []
    results_files = sorted([f for f in path.glob(f"*temp_{temp}.csv") if f.is_file()])
    for file_path in results_files:
        df = pd.read_csv(file_path, usecols=["attribute", "offset", "offset_CI", "p_offset", "tilt", "tilt_CI", "p_tilt", "r_squared_eq_1"])
        df = df.loc[:, ["attribute", "offset", "offset_CI", "p_offset", "tilt", "tilt_CI", "p_tilt", "r_squared_eq_1"]]
        df["model"] = str(file_path).split("_nbr_sim")[0].split("/")[-1]

        temp_results.append(df)
    results_table.append(pd.concat(temp_results))
results_table[0].to_csv("experiments/stats_results/results_temp_0.1.csv", index = False)
results_table[1].to_csv("experiments/stats_results/results_temp_1.0.csv", index = False)

# Check trials

In [70]:
df_gpt = pd.read_csv(get_most_recent_results(model = "gpt-5-nano", attribute = "truth")["path"])
df_gpt["rating"] = df_gpt["rating"].apply(lambda x: get_rating_string(x))
df_gpt["rating"] = pd.to_numeric(df_gpt["rating"], errors='coerce')

In [71]:
trials_stats = df_gpt.groupby(["simulation_num", "statement_id", "target"])["rating"].std().reset_index().rename(columns = {"rating": "rating_trials_std"})
plot_std(trials_stats[trials_stats["target"] == True]["rating_trials_std"], model="GPT_trails", temperature=1, attribute="truth", repeated=True)
plot_std(trials_stats[trials_stats["target"] == False]["rating_trials_std"], model="GPT_trails", temperature=1, attribute="truth", repeated=False)

In [72]:
stats.ttest_ind(trials_stats[trials_stats["target"] == False]["rating_trials_std"], trials_stats[trials_stats["target"] == True]["rating_trials_std"], nan_policy="omit")

TtestResult(statistic=np.float64(-0.19484667082369242), pvalue=np.float64(0.8455229874319248), df=np.float64(3957.0))

In [73]:
trials_stats = df.groupby(["simulation_num", "statement_id"])["rating"].std().reset_index().rename(columns={"rating": "rating_trials_std"})
plot_std(trials_stats["rating_trials_std"], model="GPT_trails", temperature=1, attribute="truth")

In [78]:
df_llama = pd.read_csv(get_most_recent_results(model = "Meta-Llama-3.1-8B-Instruct", attribute = "truth")["path"])
claims_ratings = df_llama[df_llama["iteration_num"].isin(["claim", "rating"])]
ratings = []
statement = ""
statement_id = ""
target = ""
type = ""
for line in claims_ratings.itertuples():
    if line.role == "user":
        statement = line.content
        statement_id = line.statement_id
        target = True if (line.target == True) else False
        type = line.type
    else:
        ratings.append(pd.DataFrame({"simulation_num": line.simulation_num,
                                           "trial_num": line.trial_num,
                                           "iteration_num": line.iteration_num,
                                           "statement_id": statement_id,
                                           "statement": [statement],
                                           "target": target,
                                           "type": type,
                                           "truth_rating": line.content
                                           }))
ratings = pd.concat(ratings)
ratings["truth_rating"] = ratings["truth_rating"].apply(lambda x: get_rating_string(x))
ratings["truth_rating"] = pd.to_numeric(ratings["truth_rating"], errors='coerce')
ratings_llama = ratings.reset_index(drop = True)

In [79]:
trials_stats = ratings_llama.groupby(["simulation_num", "statement_id"])["truth_rating"].std().reset_index().rename(columns={"truth_rating": "truth_rating_trials_std"})

In [80]:
for model in ["Meta-Llama-3.1-8B-Instruct", "gemma-3-4b-it", "Qwen2.5-7B-Instruct"]:
    for temp in [1.0, 0.1]:
        results = pd.read_csv(get_most_recent_results(model = model, attribute=attribute, temperature=temp)["path"], dtype={"rating": str, "truth_rating": str}).rename(columns={"truth_rating": "rating"})
        claims_ratings = results[results["iteration_num"].isin(["claim", "rating"])]
        ratings = []
        statement = ""
        statement_id = ""
        target = ""
        type = ""
        for line in claims_ratings.itertuples():
            if line.role == "user":
                statement = line.content
                statement_id = line.statement_id
                target = True if (line.target == True) else False
                type = line.type
            else:
                ratings.append(pd.DataFrame({"simulation_num": line.simulation_num,
                                                   "trial_num": line.trial_num,
                                                   "iteration_num": line.iteration_num,
                                                   "statement_id": statement_id,
                                                   "statement": [statement],
                                                   "target": target,
                                                   "type": type,
                                                   "truth_rating": line.content
                                                   }))
        ratings = pd.concat(ratings)
        ratings["truth_rating"] = ratings["truth_rating"].apply(lambda x: get_rating_string(x))
        ratings["truth_rating"] = pd.to_numeric(ratings["truth_rating"], errors='coerce')
        ratings_llama = ratings.reset_index(drop = True)
        trials_stats = ratings_llama.groupby(["simulation_num", "statement_id"])["truth_rating"].std().reset_index().rename(columns={"truth_rating": "truth_rating_trials_std"})
        print(model, "temperature ", temp)
        print(len(trials_stats[trials_stats["truth_rating_trials_std"] > 1]) / len(trials_stats))

Meta-Llama-3.1-8B-Instruct temperature  1.0
0.379
Meta-Llama-3.1-8B-Instruct temperature  0.1
0.379
gemma-3-4b-it temperature  1.0
0.379
gemma-3-4b-it temperature  0.1
0.379
Qwen2.5-7B-Instruct temperature  1.0
0.379
Qwen2.5-7B-Instruct temperature  0.1
0.379


# Prepare file for nested random effect model

In [87]:
all_results = []
for attribute in ["truth", "sentiment", "importance", "interest"]:
    results = pd.read_csv(get_most_recent_results(model="gpt-5-nano", temperature=1.0, attribute=attribute)["path"])
    results = results.drop(columns=["scenario"])
    results["temperature"] = "1.0"
    results["rating"] = pd.to_numeric(results["rating"], errors='coerce')
    all_results.append(results)

In [89]:
for model in ["Meta-Llama-3.1-8B-Instruct", "gemma-3-4b-it", "Qwen2.5-7B-Instruct"]:
    print(model)
    for temp in [1.0, 0.1]:
        print(temp)
        for attribute in ["truth", "sentiment", "importance", "interest"]:
            print(attribute)
            results = pd.read_csv(get_most_recent_results(model = model, temperature=temp, attribute=attribute)["path"])
            claims_ratings = results[results["iteration_num"].isin(["claim", "rating"])]
            ratings = []
            statement = ""
            statement_id = ""
            target = ""
            type = ""
            for line in claims_ratings.itertuples():
                if line.role == "user":
                    statement = line.content
                    statement_id = line.statement_id
                    target = True if (line.target == True) else False
                    type = line.type
                else:
                    ratings.append(pd.DataFrame({"simulation_num": line.simulation_num,
                                                       "trial_num": line.trial_num,
                                                       "statement_id": [statement_id],
                                                       "target": target,
                                                       "rating": line.content
                                                       }))
            ratings = pd.concat(ratings)
            ratings["rating"] = ratings["rating"].apply(lambda x: get_rating_string(x))
            ratings["rating"] = pd.to_numeric(ratings["rating"], errors='coerce')
            ratings["temperature"] = str(temp)
            ratings["attribute"] = attribute
            ratings["model"] = model
            all_results.append(ratings)

Meta-Llama-3.1-8B-Instruct
1.0
truth
sentiment
importance
interest
0.1
truth
sentiment
importance
interest
gemma-3-4b-it
1.0
truth
sentiment
importance
interest
0.1
truth
sentiment
importance
interest
Qwen2.5-7B-Instruct
1.0
truth
sentiment
importance
interest
0.1
truth
sentiment
importance
interest


In [90]:
all_results = pd.concat(all_results)
all_results = all_results.reset_index(drop = True)

In [91]:
all_results.dropna(subset=["rating"], inplace=True)

In [92]:
all_results.to_csv("experiments/all_results.csv", index = False)